In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import cdsapi
from pathlib import Path

# ── target directory ──────────────────────────────────────────────────────────
SAVE_DIR = Path(rf"{DATA_ROOT}/AIS_processing/weather_data/wave")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

dataset = "reanalysis-era5-single-levels"
client  = cdsapi.Client()

MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS   = [f"{d:02d}" for d in range(1, 32)]
HOURS  = [f"{h:02d}:00" for h in range(24)]

# ── download by year (resume: skip existing)────────────────────────────────
for year in range(2002, 2022):
    target = SAVE_DIR / f"era5_swh_{year}.nc"

    if target.exists():
        print(f"[SKIP]  {year}  →  {target.name} already exists")
        continue

    print(f"[QUEUE] {year}  →  {target.name} ...")
    request = {
        "product_type": ["reanalysis"],
        "variable": ["significant_height_of_combined_wind_waves_and_swell"],
        "year":  [str(year)],
        "month": MONTHS,
        "day":   DAYS,
        "time":  HOURS,
        "data_format":     "netcdf",        # ← switched to netcdf, Python-friendly
        "download_format": "unarchived",
    }

    client.retrieve(dataset, request).download(target=str(target))
    size_gb = target.stat().st_size / 1e9
    print(f"[DONE]  {year}  →  {target.name}  ({size_gb:.1f} GB)")

print("\nAll years complete.")
total_gb = sum(f.stat().st_size for f in SAVE_DIR.glob("*.nc")) / 1e9
print(f"Total size: {total_gb:.1f} GB")
